# Tennessee Eastman Process — Adaption

Adaptiert das in *tep_model* trainierte MLP an den Fehlerfallstrom (IDV 29) mit den vier Strategien Baseline, Blind (passiv), Informed (aktiv) und Combined (`drift_adaptation.py` / `drift_detection.py`). Die Läufe zwischen Resets werden jeweils mit frischem Basismodell separat adaptiert; der Informed-Detektor verwendet die in *tep_detection* getunten Parameter.

In [ ]:
import os

base_dir = os.environ.get("BASE_DIR", r"C:\git\ModelSandbox")
plot_dir = os.path.normpath(os.path.join(base_dir, "plots"))
data_dir = os.path.normpath(os.path.join(base_dir, "data", "tep"))
model_dir = os.path.normpath(os.path.join(base_dir, "models"))
results_dir = os.path.normpath(os.path.join(base_dir, "results"))
for _d in (plot_dir, data_dir, model_dir, results_dir):
    os.makedirs(_d, exist_ok=True)

FORCE_RECOMPUTE = False  # True = alle Cache-Stufen neu rechnen (Tuning + nachgelagerte Laeufe)
IS_FINAL = True  # True: schreibt zusaetzlich den stabilen, eingebundenen Stand; False: nur Archiv

In [ ]:
import numpy as np

In [ ]:
# --- Konstanten ---
SEED = 44
RUN_ID = None
SAMPLES_PER_H = 20
DRIFT_PERIOD, DRIFT_GAMMA, DRIFT_SEED = 800.0, 0.2, 44
INFORMED_DETECTOR = "DDM"
STRIDE_TUNE = 4
TUNE_WEIGHTS = (1.0, 0.1)
N_TRIALS = {"blind": 60, "informed": 20, "combined": 20}
COOLDOWN_X = 2

rng = np.random.default_rng(SEED)

## Daten und Basismodell laden

In [ ]:
import run_registry as rr
data_store = rr.tep_data_store(data_dir)
model_store = rr.tep_model_store(model_dir)
fig_store = rr.tep_plot_store(plot_dir)
res_store = rr.tep_results_store(results_dir)

_mp = model_store.latest("model")
if _mp is None:
    raise FileNotFoundError("Kein model-Artefakt. Bitte tep_model.ipynb ausfuehren.")
model_id = model_store.parse_id(_mp, "model") if RUN_ID is None else RUN_ID
_mc = model_store.load("model", run_id=model_id, rename=True)
base_model, feature_cols = _mc["model"], _mc["feature_cols"]

error_variant = {"idv": 29, "amp": 1.0}
# Gescorte Daten liegen (wie in tep_model gespeichert) im data_store
test_scored = data_store.load("data_scored_test", run_id=model_id, rename=True)
error_scored = data_store.load("data_scored_error", run_id=model_id, variant=error_variant, rename=True)
cfg = rr.RunConfig.from_document(test_scored.attrs.get("config", test_scored.attrs))

In [ ]:
import pandas as pd

X_cd = error_scored[feature_cols].to_numpy(dtype="float32")
y_cd = error_scored["cost"].to_numpy(dtype="float32")
N = len(y_cd)

reset_pos = np.flatnonzero(error_scored["reset"].to_numpy())
segments = list(zip(np.r_[0, reset_pos], np.r_[reset_pos, N]))

_free_err = (test_scored["cost"] - test_scored["y_pred"]).to_numpy(dtype=float)
NORM_MEAN = float(np.nanmean(_free_err))
NORM_STD = float(np.nanstd(_free_err)) or 1.0
print(f"Adaptionsstrom: {N} Punkte ({N / SAMPLES_PER_H:.0f} h), {len(segments)} Laeufe | "
      f"Normierungs-Baseline: mean={NORM_MEAN:.4g}, std={NORM_STD:.4g}")

## Driftsignal rekonstruieren

Python-Port von `drift_functions.c` (bitidentisch, splitmix64) wie in *tep_detection*; dient hier der Darstellung und der numerischen Auswertung.

In [ ]:
_M64 = (1 << 64) - 1

def _drift_rand(seed, idx):
    z = (seed + idx * 0x9E3779B97F4A7C15) & _M64
    z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & _M64
    z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & _M64
    z = z ^ (z >> 31)
    return (z >> 11) / 9007199254740992.0

def _drift_uniform(seed, idx, lo, hi):
    return lo + (hi - lo) * _drift_rand(seed, idx)

def _drift_boundary(k, period, gamma, seed):
    if k <= 0:
        return 0.0
    return k * period + (2.0 * _drift_rand(seed, 1000 + k) - 1.0) * gamma * 0.5 * period

def drift_signal(t, period=DRIFT_PERIOD, gamma=DRIFT_GAMMA, seed=DRIFT_SEED):
    if t <= 0.0 or period <= 0.0:
        return 0.0
    k = int(np.floor(t / period))
    while k > 0 and t < _drift_boundary(k, period, gamma, seed):
        k -= 1
    while t >= _drift_boundary(k + 1, period, gamma, seed):
        k += 1
    t0 = _drift_boundary(k, period, gamma, seed)
    t1 = _drift_boundary(k + 1, period, gamma, seed)
    c0 = 0.0 if k == 0 else _drift_uniform(seed, 2000 + k, 0.25, 1.0)
    c1 = _drift_uniform(seed, 3000 + k, -1.0, -0.25)
    return c0 + (c1 - c0) * (t - t0) / (t1 - t0)

sim_time = error_scored["time"].to_numpy(dtype=float)
cd_signal = error_variant["amp"] * np.array([drift_signal(t) for t in sim_time])

## Detektor-Konfiguration (für die informierte Adaption)

In [ ]:
import json

import tensorflow as tf
import drift_adaptation as da
from drift_adaptation import StreamingDetector

_det_params, _err_thr = {}, 3.0
_det_results = os.path.join(results_dir, "tep_detection_results.json")
if os.path.exists(_det_results):
    with open(_det_results, "r", encoding="utf-8") as _f:
        _dr = json.load(_f)
    _bp = dict(_dr["detectors"][INFORMED_DETECTOR]["best_params"])
    _err_thr = _bp.pop("err_threshold", _err_thr)
    _det_params = _bp
    print(f"Getunte {INFORMED_DETECTOR}-Parameter geladen: {_det_params}, err_threshold={_err_thr:.3f}")
else:
    print(f"{_det_results} nicht gefunden -> Standardparameter aus drift_detection.py")

def make_informed_detector():
    return StreamingDetector(INFORMED_DETECTOR, err_threshold=_err_thr, **_det_params)

## Hyperparameter-Tuning der Adaption (optional)

Getunt wird je Strategie auf dem längsten Lauf (downgesampelt), da die Adaption auch final lauf-weise mit frischem Basismodell erfolgt.

In [ ]:
CHUNK = 24 * SAMPLES_PER_H   # 1 Tag

_a0, _b0 = max(segments, key=lambda s: s[1] - s[0])
X_tune = X_cd[_a0:_b0:STRIDE_TUNE]
y_tune = y_cd[_a0:_b0:STRIDE_TUNE]

TUNE_FIXED = dict(chunk=max(24, CHUNK // STRIDE_TUNE),
                  norm_mean=NORM_MEAN, norm_std=NORM_STD, seed=SEED)

_methods = ("blind", "informed", "combined")

def _n_trials(method):
    return N_TRIALS[method] if isinstance(N_TRIALS, dict) else int(N_TRIALS)

best_adapt_params = {}
for method in _methods:
    tune_params = {"method": method, "n_trials": _n_trials(method),
                   "weights": list(TUNE_WEIGHTS), "stride_tune": STRIDE_TUNE,
                   **{k: TUNE_FIXED[k] for k in sorted(TUNE_FIXED)}}
    tune_cfg = cfg.compose(tuning=tune_params)
    if model_store.exists("tuning", tune_cfg) and not FORCE_RECOMPUTE:
        best_adapt_params[method] = model_store.load("tuning", tune_cfg)
        continue
    tf.keras.utils.set_random_seed(SEED)
    _fac = make_informed_detector if method in ("informed", "combined") else None
    _nt = _n_trials(method)
    _study, _rmse_base = da.tune_adaptation(
        method, base_model, X_tune, y_tune, fixed=TUNE_FIXED,
        detector_factory=_fac, n_trials=_nt, weights=TUNE_WEIGHTS, seed=SEED)
    best_adapt_params[method] = da.decode_adapt_params(_study.best_params)
    _a = _study.best_trial.user_attrs
    print(f"{method:9s} score={_study.best_value:.3f} | rmse={_a['rmse']:.3f} "
          f"n_train={_a['n_train']:3d} trials={_nt} -> {_study.best_params}")
    model_store.save(best_adapt_params[method], "tuning", tune_cfg)

## Adaption ausführen

In [ ]:
ADAPT_CFG = dict(
    lr=3e-3,
    epochs=9,
    freeze=(True, False, False),
    reset=False,
    error_window=200,
    error_threshold=1.5,
    window_prev=CHUNK,
    window_post=CHUNK,
    chunk=CHUNK,
    norm_mean=NORM_MEAN, norm_std=NORM_STD,
)

STRATEGIES = {
    "baseline": dict(mode="baseline"),
    "blind":    dict(mode="blind"),
    "informed": dict(mode="informed"),
    "combined": dict(mode="combined"),
}

_RESCALE = float(STRIDE_TUNE)
_SCALE_KEYS = ("error_window", "window_prev", "window_post")

def _final_cfg(strategy):
    _c = dict(ADAPT_CFG)
    for _k, _v in best_adapt_params.get(strategy, {}).items():
        _c[_k] = max(1, int(round(_v * _RESCALE))) if _k in _SCALE_KEYS else _v
    return _c

applied_cfg = {_name: _final_cfg(_name) for _name in STRATEGIES}

_op_change = error_scored["mode"].diff().fillna(0).ne(0)
T_stat = N / max(1, int(_op_change.sum()))
COOLDOWN = int(round(COOLDOWN_X * T_stat))
print(f"T_stat ~ {T_stat:.0f} Punkte (~{T_stat / SAMPLES_PER_H:.1f} h) | "
      f"COOLDOWN = {COOLDOWN} Punkte (x={COOLDOWN_X})")

adapt_cfg = cfg.compose(adaptation={"cooldown_x": COOLDOWN_X,
                        "applied": {k: {kk: v for kk, v in applied_cfg[k].items()
                                        if kk not in ("norm_mean", "norm_std")}
                                    for k in STRATEGIES}})
if model_store.exists("adaptation", adapt_cfg) and not FORCE_RECOMPUTE:
    results = model_store.load("adaptation", adapt_cfg, rename=True)
else:
    results = {}
    for _name, _kw in STRATEGIES.items():
        tf.keras.utils.set_random_seed(SEED)
        _detector = make_informed_detector() if _kw["mode"] in ("informed", "combined") else None
        preds = np.empty(N, np.float32)
        errors = np.empty(N, np.float32)
        errors_norm = np.empty(N, np.float32)
        train_steps, detect_steps = [], []
        for _a, _b in segments:
            _res = da.run_adaptation(base_model, X_cd[_a:_b], y_cd[_a:_b],
                                     detector=_detector, seed=SEED,
                                     cooldown=COOLDOWN, **_kw, **applied_cfg[_name])
            preds[_a:_b] = _res["preds"]
            errors[_a:_b] = _res["errors"]
            errors_norm[_a:_b] = _res["errors_norm"]
            train_steps += [int(_a + s) for s in _res["train_steps"]]
            detect_steps += [int(_a + s) for s in _res["detect_steps"]]
        results[_name] = dict(preds=preds, errors=errors, errors_norm=errors_norm,
                              train_steps=train_steps, detect_steps=detect_steps)
        print(f"{_name:9s}: {len(train_steps):3d} Nachtrainings, {len(detect_steps):3d} Detektionen")
    model_store.save(results, "adaptation", adapt_cfg)

## Vergleich der Adaptionsstrategien

Die folgenden Abbildungen zeigen je Strategie den normierten Modellfehler, das rekonstruierte Driftsignal (an den Resets unterbrochen) und die Detektionszeitpunkte der informierten Strategien.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from src.utils import thesis_style as ts


t_h = np.arange(N) / SAMPLES_PER_H
ALPHA, S, FRAC = 0.5, 6, 0.2
C_DRIFT, C_DET = ts.C["drift_signal"], ts.C["detection"]
rng_plot = np.random.default_rng(SEED)

PANELS = [("baseline", "Baseline"), ("blind", "Blind"),
          ("informed", "Informed"), ("combined", "Combined")]

def align_zero(ax_left, ax_right):
    """0 beider Ordinaten auf gleiche Hoehe (wie in den uebrigen Notebooks)."""
    ax_left.figure.canvas.draw()
    lims = [ax_left.get_ylim(), ax_right.get_ylim()]
    fracs = [(-lo / (hi - lo) if hi != lo else 0.5) for lo, hi in lims]
    r = min(max(max(fracs), 1e-3), 1 - 1e-3)
    for ax, (lo, hi) in zip((ax_left, ax_right), lims):
        span = max(hi / (1 - r), -lo / r)
        ax.set_ylim(-r * span, (1 - r) * span)

cut = reset_pos
for k, (key, title) in enumerate(PANELS):
    err_norm = results[key]["errors_norm"]
    detect = results[key]["detect_steps"]

    fig, ax = plt.subplots(figsize=(ts.fig_width(), ts.fig_width(0.33)))
    m = rng_plot.random(N) < FRAC
    c_err = ts.C["drift_afflicted"] if key == "baseline" else ts.C["adapted"]
    ax.scatter(t_h[m], err_norm[m], color=c_err, s=S, alpha=ALPHA,
               zorder=3, rasterized=True)

    ax2 = ax.twinx()
    for a, b in zip(np.r_[0, cut], np.r_[cut, N]):
        ax2.plot(t_h[a:b], cd_signal[a:b], **ts.line("drift_signal", linewidth=1.5))
    ax2.set_ylabel("Driftsignal $c(t)$")
    ax2.spines["right"].set_visible(True)
    ax2.spines["top"].set_visible(False)

    for d in detect:
        ax.axvline(t_h[d], zorder=2, **ts.vline("detection", lw=0.7, linestyle="--"))

    ax.set_ylabel("Norm. Fehler")
    ax.margins(x=0)
    ax.grid(True, alpha=0.2)
    if k == len(PANELS) - 1:
        ax.set_xlabel("Zeit $t$ [h]")

    if k == 0:
        handles = [Line2D([0], [0], marker="o", linestyle="", color=ts.C["drift_afflicted"],
                          alpha=ALPHA, label="drift-behaftet"),
                   Line2D([0], [0], marker="o", linestyle="", color=ts.C["adapted"],
                          alpha=ALPHA, label="adaptiert"),
                   Line2D([0], [0], color=C_DRIFT, label="Driftsignal $c(t)$"),
                   Line2D([0], [0], color=C_DET, linestyle="--", label="Drift erkannt")]
        ax.legend(handles=handles, loc="lower left",
                  ncol=2, frameon=False, columnspacing=1.4, handletextpad=0.5)

    align_zero(ax, ax2)
    fig.tight_layout()
    fig_store.save_figure(fig, f"tep_adaptation_{key}", adapt_cfg, final=IS_FINAL, savefig_kwargs={"dpi": 300}, archive_kwargs={"dpi": 300}, variant=error_variant)
    plt.show()

## RMSE-Verlauf der Strategien

Gleitender RMSE (Fenster 24 h) aller Strategien über der Zeit, mit dem Driftsignal auf der zweiten Ordinate.

In [ ]:
from src.utils import thesis_style as ts

WIN = 24 * SAMPLES_PER_H

fig, ax = plt.subplots(figsize=(ts.fig_width(), ts.fig_width(0.45)))

_adapt_labeled = False
for key, title in PANELS:
    e = pd.Series(results[key]["errors"])
    rmse_roll = np.sqrt((e ** 2).rolling(WIN, min_periods=1).mean())
    if key == "baseline":
        _col, _lab = ts.C["drift_afflicted"], "drift-behaftet"
    else:
        _col = ts.C["adapted"]
        _lab = "adaptiert" if not _adapt_labeled else None
        _adapt_labeled = True
    ax.plot(t_h, rmse_roll.values, color=_col, linewidth=1.4, label=_lab)

ax2 = ax.twinx()
for a, b in zip(np.r_[0, cut], np.r_[cut, N]):
    ax2.plot(t_h[a:b], cd_signal[a:b], **ts.line("drift_signal", linewidth=1.0))
ax2.set_ylabel("Driftsignal $c(t)$")
ax2.spines["right"].set_visible(True)
ax2.spines["top"].set_visible(False)

ax.set_ylabel("Gleitender RMSE")
ax.set_xlabel("Zeit $t$ [h]")
ax.margins(x=0)
ax.grid(True, alpha=0.2)
ax.legend(loc="upper left", ncol=2, frameon=False)
fig.tight_layout()
plt.show()

## Numerische Auswertung

In [ ]:
def _strategy_stats(res, c, rmse_baseline):
    e = np.asarray(res["errors"], float)
    c = np.asarray(c, float)
    rmse = float(np.sqrt(np.mean(e ** 2)))
    return dict(
        rmse=rmse,
        bias=float(np.mean(e)),
        sigma=float(np.std(e)),
        corr_abs=float(np.corrcoef(np.abs(e), np.abs(c))[0, 1]),
        n_train=int(len(res["train_steps"])),
        n_detect=int(len(res["detect_steps"])),
        rmse_improvement=float(1.0 - rmse / rmse_baseline) if rmse_baseline else 0.0,
    )

_rmse_base = float(np.sqrt(np.mean(np.asarray(results["baseline"]["errors"], float) ** 2)))
stats = pd.DataFrame({k: _strategy_stats(results[k], cd_signal, _rmse_base)
                      for k, _ in PANELS}).T
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
print(stats.to_string())

## Ergebnisse speichern

In [ ]:
from src.utils import results_export as rx

_keys = [(k, k, "int" if k in ("n_train", "n_detect") else "num")
         for k in ["rmse", "bias", "sigma", "corr_abs",
                   "n_train", "n_detect", "rmse_improvement"]]

(rx.ResultDoc()
   .integer("n_samples", N)
   .set("samples_per_hour", SAMPLES_PER_H)
   .integer("n_runs", len(segments))
   .set("reset_indices", reset_pos)
   .set("informed_detector", INFORMED_DETECTOR)
   .set("config", ADAPT_CFG)
   .set("tuning", {
       "stride_tune": int(STRIDE_TUNE),
       "n_trials": (N_TRIALS if isinstance(N_TRIALS, dict) else int(N_TRIALS)),
       "weights": list(TUNE_WEIGHTS),
       "best_params": best_adapt_params,
   })
   .set("cooldown", {"x": COOLDOWN_X, "points": int(COOLDOWN), "t_stat_points": float(T_stat)})
   .set("applied_config", applied_cfg)
   .stats(stats, _keys, into="strategies", rows=[k for k, _ in PANELS])
   .save(res_store, "tep_adaptation_results", adapt_cfg, final=IS_FINAL,
         variant=error_variant, parents={"model": model_id}))

In [ ]:
def _de(x, nd):
    if x is None or (isinstance(x, float) and x != x):
        return "--"
    return f"{x:.{nd}f}".replace(".", "{,}")

_name_map = {"baseline": "Baseline", "blind": "Blind",
             "informed": "Informed", "combined": "Combined"}
_keys = [("rmse", "Rmse", 3), ("bias", "Bias", 3), ("sigma", "Sigma", 3),
         ("corr_abs", "CorrAbs", 2), ("rmse_improvement", "Improve", 3)]

_lines = ["% automatisch erzeugt aus tep_adaptation.ipynb - nicht manuell editieren"]
for k, _ in PANELS:
    m = _name_map[k]
    for col, suf, nd in _keys:
        _lines.append(rf"\newcommand{{\tepadapt{m}{suf}}}{{{_de(stats.loc[k, col], nd)}}}")
    _lines.append(rf"\newcommand{{\tepadapt{m}NTrain}}{{{int(stats.loc[k, 'n_train'])}}}")
    _lines.append(rf"\newcommand{{\tepadapt{m}NDetect}}{{{int(stats.loc[k, 'n_detect'])}}}")
_tex = "\n".join(_lines) + "\n"
_pt, _ = res_store.save_text(_tex, "tep_adaptation_stats", adapt_cfg, final=IS_FINAL, variant=error_variant)
print(f"LaTeX-Makros -> {_pt}\n")
print(_tex)